In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
import random
from pathlib import Path
import os

In [11]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
K_FOLDS = 5

In [12]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Treinando em:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Treinando em: cpu


In [14]:
def load_data_from_folders(dir_data, class_names, reshape_type):
    data_list = []
    for class_index, class_name in enumerate(class_names):
        dir_class = Path(dir_data) / class_name / reshape_type

        if dir_class.exists():
            images = list(dir_class.glob('*.*'))
            print(f"Imagens encontradas em {class_name}: {len(images)}")
            for img_path in images:
                data_list.append((str(img_path), class_index))
        else:
            print(f"AVISO: Diretório {dir_class} não encontrado!!!!")

    return data_list

In [15]:
class MedicalImageDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [16]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [17]:
def train_one_fold(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels)
            total += labels.size(0)

        train_acc = correct.double() / total

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.numpy())

        f1 = f1_score(val_labels, val_preds, average='macro')
        acc = accuracy_score(val_labels, val_preds)

        print(f"Época {epoch+1}/{EPOCHS} | TrainAcc {train_acc:.4f} | ValAcc {acc:.4f} | ValF1 {f1:.4f}")

    return acc, f1

In [18]:
def run_kfold(dataset_path, dataset_type, class_names):

    data_list = load_data_from_folders(dataset_path, class_names, dataset_type)
    print(f"Total de imagens: {len(data_list)}")

    paths = [x[0] for x in data_list]
    labels = [x[1] for x in data_list]

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(paths, labels)):
        print("\n============================")
        print(f"FOLD {fold+1}/{K_FOLDS}")
        print("============================")

        train_data = [(paths[i], labels[i]) for i in train_idx]
        val_data = [(paths[i], labels[i]) for i in val_idx]

        train_dataset = MedicalImageDataset(train_data, transform=transform)
        val_dataset = MedicalImageDataset(val_data, transform=transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        # modelo pré-treinado
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.last_channel, len(class_names))
        model = model.to(DEVICE)

        acc, f1 = train_one_fold(model, train_loader, val_loader)

        fold_results.append({
            'fold': fold+1,
            'accuracy': acc,
            'f1_score': f1
        })

    df = pd.DataFrame(fold_results)
    df.to_csv(f'resultados_kfold_{dataset_type}.csv', index=False)
    print("\n===== RESULTADOS FINAIS DO K-FOLD =====")
    print(df)
    print("\nMédias:")
    print(df.mean())

    return df

In [21]:
path = '..//feature_to_image//saida'
classes = ['healthy', 'severe']

df_fclassical = run_kfold(path, 'F-Classical', classes)
df_frecplot = run_kfold(path, 'F-RecPlot', classes)

Imagens encontradas em healthy: 114
Imagens encontradas em severe: 114
Total de imagens: 228

FOLD 1/5
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\danbo/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:03<00:00, 4.70MB/s]


KeyboardInterrupt: 